In [1]:
import pandas as pd
import numpy as np

# Read the Liga F all fixtures CSV file
liga_f_df = pd.read_csv('data/spain/liga_f_all_fixtures_2025-10-31.csv')
liga_f_df['home_score'] = liga_f_df['home_score'].fillna(0.0)
liga_f_df['away_score'] = liga_f_df['away_score'].fillna(0.0)

print("Original dataset shape:", liga_f_df.shape)
print("\nAvailable seasons:")
print(liga_f_df['season'].value_counts().sort_index())

# Filter for 2022-2023 season only
liga_f_2022_2023 = liga_f_df[liga_f_df['season'] == '2022-2023'].copy()
liga_f_2023_2024 = liga_f_df[liga_f_df['season'] == '2023-2024'].copy()
liga_f_2024_2025 = liga_f_df[liga_f_df['season'] == '2024-2025'].copy()

print(f"\nFiltered dataset for 2022-2023 season shape: {liga_f_2022_2023.shape}")
print(f"Number of matches: {len(liga_f_2022_2023)}")

liga_f_2022_2023.head()

Original dataset shape: (960, 20)

Available seasons:
season
2022-2023    240
2023-2024    240
2024-2025    240
2025-2026    240
Name: count, dtype: int64

Filtered dataset for 2022-2023 season shape: (240, 20)
Number of matches: 240


,season,gameweek,date,day_of_week,start_time,home_team,away_team,home_score,away_score,score,home_xg,away_xg,venue,attendance,referee,status,result,home_team_url,away_team_url,match_report_url
720,2022-2023,2,2022-09-17,Sat,12:00 (03:00),Barcelona,Tenerife,2.0,0.0,2–0,1.9,0.4,Estadi Johan Cruyff,NaN,María Dolores Martínez Madrona,completed,home_win,https://fbref.com/en/squads/15f49df1/2022-2023...,https://fbref.com/en/squads/4c088abe/2022-2023...,https://fbref.com/en/matches/4df3a732/Barcelon...
721,2022-2023,2,2022-09-17,Sat,12:00 (03:00),Alavés,Madrid CFF,1.0,2.0,1–2,1.1,1.2,Ciudad Deportiva José Luis Compañón,NaN,Manuel Pascali,completed,away_win,https://fbref.com/en/squads/aa11fb42/Alaves-Wo...,https://fbref.com/en/squads/89818574/2022-2023...,https://fbref.com/en/matches/87c755cd/Alaves-M...
722,2022-2023,2,2022-09-17,Sat,16:00 (07:00),Real Madrid,Valencia,2.0,0.0,2–0,1.6,0.8,Estadio Alfredo Di Stéfano,NaN,Marta Huerta de Aza,completed,home_win,https://fbref.com/en/squads/54582b93/2022-2023...,https://fbref.com/en/squads/f96ff499/2022-2023...,https://fbref.com/en/matches/d0329f46/Real-Mad...
723,2022-2023,2,2022-09-17,Sat,16:00 (07:00),Real Sociedad,Villarreal,2.0,0.0,2–0,0.7,0.3,Instalaciones de Zubieta,NaN,Alicia Espinosa Ríos,completed,home_win,https://fbref.com/en/squads/c21f25d3/2022-2023...,https://fbref.com/en/squads/7a7bef84/2022-2023...,https://fbref.com/en/matches/abfde9d9/Real-Soc...
724,2022-2023,2,2022-09-17,Sat,18:00 (09:00),Sevilla,Atlético Madrid,1.0,3.0,1–3,1.1,1.4,Estadio Jesús Navas,NaN,Bruno Gallo,completed,away_win,https://fbref.com/en/squads/215d9026/2022-2023...,https://fbref.com/en/squads/b56c2667/2022-2023...,https://fbref.com/en/matches/f4452586/Sevilla-...


In [2]:
# Split the filtered dataset into training and testing sets
train_liga_f_2022_2023 = liga_f_2022_2023.iloc[:len(liga_f_2022_2023)//2]
train_liga_f_2023_2024 = liga_f_2023_2024.iloc[:len(liga_f_2023_2024)//2]
train_liga_f_2024_2025 = liga_f_2024_2025.iloc[:len(liga_f_2024_2025)//2]

# =========================
# === VALIDATION BLOCK ===
# =========================

# Hardcoded ground truth rankings (replace with real ones)
# --------------------------------------- WOMEN ------------------------------------------------------------------------
trueLigaF2223 = [
    'Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Tenerife', 'Sevilla', 'Real Sociedad',
    'Valencia', 'Athletic Club', 'Levante Planas', 'Real Betis', 'Sporting Huelva', 'Villarreal', 'Alhama', 'Alavés'
  ]
trueLigaF2324 = [
    'Barcelona', 'Real Madrid', 'Atlético Madrid', 'Levante', 'Madrid CFF', 'Athletic Club', 'Sevilla', 'Real Sociedad',
    'Tenerife', 'Eibar', 'Real Betis', 'Valencia', 'Levante Planas', 'Granada', 'Villarreal', 'Sporting Huelva'
  ]
trueLigaF2425 = [
    'Barcelona', 'Real Madrid', 'Atlético Madrid', 'Athletic Club', 'Granada', 'Tenerife', 'Real Sociedad', 'Eibar',
    'Sevilla', 'Madrid CFF', 'Espanyol', 'Levante', 'Lavante Badalona', 'Dep La Coruña', 'Valencia', 'Real Betis'
  ]
# Run model
# Make sure you load your `league` data before this point
# team_ratings, sorted_teams = Colley(league, weighting=0)

# Example only: to use for validation once the Colley function returns sorted_teams
def validate_ranking(sorted_teams, rankingTrueNames):
    # Get top predicted 16 teams
    rankingTrainNames = sorted_teams

    # Convert rankingTrueNames to predicted ranks
    rankingTrainIndices = []
    for name in rankingTrueNames:
        if name in sorted_teams:
            rankingTrainIndices.append(sorted_teams.index(name) + 1)  # 1-based index
        else:
            rankingTrainIndices.append(len(sorted_teams))  # assume worst if not found

    # Use the full list of teams for comparison
    rankingTrueIndices = list(range(1, len(rankingTrueNames) + 1))  # True positions: 1 to len(rankingTrueNames)

    # Error vector
    errorVector = [abs(rt - rp) for rt, rp in zip(rankingTrueIndices, rankingTrainIndices)]
    errorTrain = np.linalg.norm(errorVector)
    percentDiff = [ev / rt for ev, rt in zip(errorVector, rankingTrueIndices)]
    meanPercentDiff = np.mean(percentDiff)


    return errorTrain, meanPercentDiff

# Example usage (assuming you already have `league` dataframe):
# team_ratings, sorted_teams = Colley(league, weighting=0)
# validate_ranking(sorted_teams, rankingTrueNames)

In [3]:
# Define the range for home win coefficients
home_win_coeffs = np.arange(0.25, 1.76, 0.25)

# Define the range for away win coefficients
away_win_coeffs = np.arange(0.25, 1.76, 0.25)

# Define the range for away win coefficients
margin_coeffs = np.arange(0.0, 1.01, 0.50)

# Define the range for date  coefficients
date_coeffs = np.arange(0, 1.01, 0.5)

print("Home Win Coefficients:", home_win_coeffs)
print("Away Win Coefficients:", away_win_coeffs)
print("Margin Coefficients:", margin_coeffs)
print("Date Coefficients:", date_coeffs)

Home Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Away Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Margin Coefficients: [0.  0.5 1. ]
Date Coefficients: [0.  0.5 1. ]


In [4]:
best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
      for date_c in date_coeffs:
        for margin_c in margin_coeffs:
        # The actual running of Colley and validation will happen in the next subtask
           pass # Placeholder for the next step

In [ ]:
import numpy as np
import pandas as pd

# Define the Colley function, specifically for weighting mode 4 optimization
def Colley_weighted_optimized(league, home_win_coeff, away_win_coeff, margin_coeff, date_coeff):
    hTeam = league['home_team']
    aTeam = league['away_team']

    teams = sorted(set(hTeam).union(set(aTeam)))
    A = 2 * np.eye(len(teams))
    b = np.ones(len(teams))
    teamIndex = {team: idx for idx, team in enumerate(teams)}

    # Margin weights
    def margin(s):
        s = str(s).replace('–', '-')
        try:
            x, y = [int(t) for t in s.split('-')]
            diff = abs(x - y)
            return min(3, (1 + diff**margin_coeff) / 2)
        except:
            return 1
    
    w_margin = league['score'].apply(margin).to_numpy()
    w_margin_norm = w_margin / np.mean(w_margin)  # Normalize to mean=1

    # Home/Away weights
    w_home_away = np.ones(len(league))
    for i in range(len(league)):
        row = league.iloc[i]
        score_txt = str(row['score']).replace('–', '-')
        try:
            hScore, aScore = [int(t) for t in score_txt.split('-')]
            if aScore > hScore:
                w_home_away[i] = away_win_coeff
            elif hScore > aScore:
                w_home_away[i] = home_win_coeff
        except:
            continue

    # Date weights
    latest_week = league['gameweek'].max()
    w_date_raw = (league['gameweek'] / latest_week).to_numpy() ** date_coeff
    w_date = w_date_raw / np.mean(w_date_raw)  # Normalize to mean=1

    # Combine weights
    w = w_margin_norm * w_home_away * w_date

    # Build Colley matrix
    for i in range(len(league)):
        row = league.iloc[i]
        home, away = row['home_team'], row['away_team']
        score_txt = str(row['score']).replace('–', '-')

        try:
            hScore, aScore = [int(t) for t in score_txt.split('-')]
        except:
            continue

        hi = teamIndex[home]
        ai = teamIndex[away]

        # Update Colley matrix
        A[hi, ai] -= w[i]
        A[ai, hi] -= w[i]
        A[hi, hi] += w[i]
        A[ai, ai] += w[i]

        # Update vector b
        if hScore > aScore:
            b[hi] += 0.5 * w[i]
            b[ai] -= 0.5 * w[i]
        elif aScore > hScore:
            b[ai] += 0.5 * w[i]
            b[hi] -= 0.5 * w[i]
        # ties: no change

    # Solve for ratings
    r = np.linalg.solve(A, b)
    team_ratings = {team: r[idx] for idx, team in enumerate(teams)}
    sorted_teams = sorted(team_ratings, key=team_ratings.get, reverse=True)
    
    return team_ratings, sorted_teams

history = []

In [6]:
# Test with doubled coefficients
ratings1, a1 = Colley_weighted_optimized(train_liga_f_2022_2023, 0.25, 0.25, 1.0, 0.0)
ratings2, a2 = Colley_weighted_optimized(train_liga_f_2022_2023, 0.5, 0.5, 1.0, 0.0)  # 2x scale

# Check if rankings are identical
print("Rankings identical?", a1 == a2)

# Compare
print(np.allclose(list(ratings1.values()), list(ratings2.values())))

Rankings identical? False
False


In [7]:
# WSL 2022/23
best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_date_coeff = None
best_margin_coeff = None

try_num = 0

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
                # Run Colley with the current coefficients
                team_ratings, sorted_teams = Colley_weighted_optimized(train_liga_f_2022_2023, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2223)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2022/23',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
               })

               # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2223)
                    print("--------------------------------------------------------------------------------")


print("\nColley Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=7.7460, Mean%Diff=0.1389
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Real Sociedad', 'Valencia', 'Sporting Huelva', 'Sevilla', 'Tenerife', 'Levante Planas', 'Athletic Club', 'Real Betis', 'Villarreal', 'Alavés', 'Alhama']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Tenerife', 'Sevilla', 'Real Sociedad', 'Valencia', 'Athletic Club', 'Levante Planas', 'Real Betis', 'Sporting Huelva', 'Villarreal', 'Alhama', 'Alavés']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.50, Date Coeff=0.00, Error=7.6158, Mean%Diff=0.1371
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Real Sociedad', 'Valencia', 'Sevilla', 'Sporting Huelva', 'Athletic Club', 'Teneri

In [8]:
# WSL 2023/24
try_num = 0

best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_date_coeff = None
best_margin_coeff = None


for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:           
                # Run Colley with the current coefficients
                team_ratings, sorted_teams = Colley_weighted_optimized(train_liga_f_2023_2024, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2324)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2023/24',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
               })

                # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2324)
                    print("--------------------------------------------------------------------------------")


print("\nColley Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=7.3485, Mean%Diff=0.1560
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Madrid CFF', 'Levante', 'Sevilla', 'Real Sociedad', 'Tenerife', 'Athletic Club', 'Levante Planas', 'Valencia', 'Villarreal', 'Real Betis', 'Eibar', 'Granada', 'Sporting Huelva']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Levante', 'Madrid CFF', 'Athletic Club', 'Sevilla', 'Real Sociedad', 'Tenerife', 'Eibar', 'Real Betis', 'Valencia', 'Levante Planas', 'Granada', 'Villarreal', 'Sporting Huelva']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.50, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=7.0711, Mean%Diff=0.1510
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Madrid CFF', 'Levante', 'Sevilla', 'Real Sociedad', 'Tenerife', 'Levante Planas', 'Athletic Club', 'Valenci

In [9]:
# WSL 2024/25
try_num = 0

best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_date_coeff = None
best_margin_coeff = None


for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
            # Run Colley with the current coefficients
                team_ratings, sorted_teams = Colley_weighted_optimized(train_liga_f_2024_2025, home_coeff, away_coeff, margin_coeff, date_coeff)

            # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2425)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2024/25',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
                })

                # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_date_coeff = date_coeff
                    best_margin_coeff = margin_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2425)
                    print("--------------------------------------------------------------------------------")


print("\nColley Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=8.6023, Mean%Diff=0.1925
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Real Sociedad', 'Atlético Madrid', 'Athletic Club', 'Granada', 'Tenerife', 'Levante Badalona', 'Eibar', 'Espanyol', 'Madrid CFF', 'Real Betis', 'Sevilla', 'Dep La Coruña', 'Levante', 'Valencia']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Athletic Club', 'Granada', 'Tenerife', 'Real Sociedad', 'Eibar', 'Sevilla', 'Madrid CFF', 'Espanyol', 'Levante', 'Lavante Badalona', 'Dep La Coruña', 'Valencia', 'Real Betis']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.50, Date Coeff=0.00, Error=7.8740, Mean%Diff=0.1669
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Real Sociedad', 'Athletic Club', 'Granada', 'Tenerife', 'Levante Badalona', 'Espanyol', 'Madrid CFF', 'Eibar', 

In [10]:
hist_df = pd.DataFrame(history)
hist_df.head(5)
# print(hist_df.tail(5))

,try,seasons,home_coeff,away_coeff,margin_coeff,date_coeff,Error,Mean%Diff
0,1,2022/23,0.25,0.25,0.0,0.0,7.745967,0.138857
1,2,2022/23,0.25,0.25,0.5,0.0,7.615773,0.137090
2,3,2022/23,0.25,0.25,1.0,0.0,6.324555,0.123606
3,4,2022/23,0.25,0.25,0.0,0.5,9.055385,0.165011
4,5,2022/23,0.25,0.25,0.5,0.5,8.717798,0.172236


In [11]:
from scipy import stats

vars_to_mode = ["home_coeff", "away_coeff", "margin_coeff", "date_coeff"]

def mode_at_min_mpd(df):
    min_mpd = df["Mean%Diff"].min()
    best_rows = df[df["Mean%Diff"] == min_mpd]
    
    mode_values = {}
    for col in vars_to_mode:
        m = best_rows[col].mode()
        # if multiple modes, take the first (deterministic)
        mode_values[col] = float(m.iloc[0])
    
    return pd.Series(mode_values)

# per-season
season_result = (
    hist_df
    .groupby("seasons")
    .apply(mode_at_min_mpd)
)

# ALL seasons
all_result = mode_at_min_mpd(hist_df).to_frame(name="ALL").T

# same orientation as before
final_table = pd.concat([season_result, all_result]).T

final_table


C:\Users\Naufal\AppData\Local\Temp\ipykernel_17852\2341830475.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


,2022/23,2023/24,2024/25,ALL
home_coeff,0.25,1.5,0.25,1.5
away_coeff,0.25,0.5,0.25,0.5
margin_coeff,1.00,0.5,0.50,0.5
date_coeff,0.00,0.0,0.00,0.0


In [12]:
from scipy import stats

vars_to_mode = ["home_coeff", "away_coeff", "margin_coeff", "date_coeff"]

def mode_at_min_mpd(df):
    min_mpd = df["Error"].min()
    best_rows = df[df["Error"] == min_mpd]
    
    mode_values = {}
    for col in vars_to_mode:
        m = best_rows[col].mode()
        # if multiple modes, take the first (deterministic)
        mode_values[col] = float(m.iloc[0])
    
    return pd.Series(mode_values)

# per-season
season_result = (
    hist_df
    .groupby("seasons")
    .apply(mode_at_min_mpd)
)

# ALL seasons
all_result = mode_at_min_mpd(hist_df).to_frame(name="ALL").T

# same orientation as before
final_table = pd.concat([season_result, all_result]).T

final_table

C:\Users\Naufal\AppData\Local\Temp\ipykernel_17852\2670775696.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


,2022/23,2023/24,2024/25,ALL
home_coeff,0.25,1.5,0.5,0.25
away_coeff,0.25,0.5,0.5,0.50
margin_coeff,1.00,0.0,1.0,0.00
date_coeff,0.00,0.0,0.0,0.00


In [13]:
hist_df.to_csv("resultsColley.csv", index=False)

In [14]:
import altair as alt

hist_df = pd.DataFrame(history)

# best try per season (min Error, tie-break min Mean%Diff)
best_df = (hist_df.sort_values(["seasons", "Error", "Mean%Diff"])
           .groupby("seasons", as_index=False)
           .first()[["seasons", "home_coeff"]])

charts = []
for season in hist_df['seasons'].unique():
    season_df = hist_df[hist_df['seasons'] == season]
    season_best = best_df[best_df['seasons'] == season]
    
    base = alt.Chart(season_df).encode(
        x=alt.X("home_coeff:Q", title="home_coeff")
    )
    
    error_line = base.mark_boxplot().encode(
        y=alt.Y("Error:Q", scale=alt.Scale(zero=False), title="Error")
    )
    
    mpd_line = base.mark_boxplot().encode(
        y=alt.Y("Mean%Diff:Q", scale=alt.Scale(zero=False), title="Mean % Difference")
    )
    
    rules = alt.Chart(season_best).mark_rule(strokeDash=[6,4], color='red').encode(
        x=alt.X("home_coeff:Q", scale=alt.Scale(zero=False))
    )
    
    season_chart = alt.vconcat(
        (error_line + rules).properties(width=300, height=200, title=f"Season {season} - Error"),
        (mpd_line + rules).properties(width=300, height=200, title=f"Season {season} - Mean % Diff")
    )
    
    charts.append(season_chart)

chart = alt.hconcat(*charts)
chart

alt.HConcatChart(...)

In [15]:
import altair as alt

hist_df = pd.DataFrame(history)

# best try per season (min Error, tie-break min Mean%Diff)
best_df = (hist_df.sort_values(["seasons", "Error", "Mean%Diff"])
           .groupby("seasons", as_index=False)
           .first()[["seasons", "away_coeff"]])

charts = []
for season in hist_df['seasons'].unique():
    season_df = hist_df[hist_df['seasons'] == season]
    season_best = best_df[best_df['seasons'] == season]
    
    base = alt.Chart(season_df).encode(
        x=alt.X("away_coeff:Q", title="away_coeff")
    )
    
    error_line = base.mark_boxplot().encode(
        y=alt.Y("Error:Q", scale=alt.Scale(zero=False), title="Error")
    )
    
    mpd_line = base.mark_boxplot().encode(
        y=alt.Y("Mean%Diff:Q", scale=alt.Scale(zero=False), title="Mean % Difference")
    )
    
    rules = alt.Chart(season_best).mark_rule(strokeDash=[6,4], color='red').encode(
        x=alt.X("away_coeff:Q", scale=alt.Scale(zero=False))
    )
    
    season_chart = alt.vconcat(
        (error_line + rules).properties(width=300, height=200, title=f"Season {season} - Error"),
        (mpd_line + rules).properties(width=300, height=200, title=f"Season {season} - Mean % Diff")
    )
    
    charts.append(season_chart)

chart = alt.hconcat(*charts)
chart

alt.HConcatChart(...)

In [16]:
import altair as alt

hist_df = pd.DataFrame(history)

# best try per season (min Error, tie-break min Mean%Diff)
best_df = (hist_df.sort_values(["seasons", "Error", "Mean%Diff"])
           .groupby("seasons", as_index=False)
           .first()[["seasons", "margin_coeff"]])

charts = []
for season in hist_df['seasons'].unique():
    season_df = hist_df[hist_df['seasons'] == season]
    season_best = best_df[best_df['seasons'] == season]
    
    base = alt.Chart(season_df).encode(
        x=alt.X("margin_coeff:Q", title="margin_coeff")
    )
    
    error_line = base.mark_boxplot().encode(
        y=alt.Y("Error:Q", scale=alt.Scale(zero=False), title="Error")
    )
    
    mpd_line = base.mark_boxplot().encode(
        y=alt.Y("Mean%Diff:Q", scale=alt.Scale(zero=False), title="Mean % Difference")
    )
    
    rules = alt.Chart(season_best).mark_rule(strokeDash=[6,4], color='red').encode(
        x=alt.X("margin_coeff:Q", scale=alt.Scale(zero=False))
    )
    
    season_chart = alt.vconcat(
        (error_line + rules).properties(width=300, height=200, title=f"Season {season} - Error"),
        (mpd_line + rules).properties(width=300, height=200, title=f"Season {season} - Mean % Diff")
    )
    
    charts.append(season_chart)

chart = alt.hconcat(*charts)
chart

alt.HConcatChart(...)

In [17]:
import altair as alt

hist_df = pd.DataFrame(history)

# best try per season (min Error, tie-break min Mean%Diff)
best_df = (hist_df.sort_values(["seasons", "Error", "Mean%Diff"])
           .groupby("seasons", as_index=False)
           .first()[["seasons", "date_coeff"]])

charts = []
for season in hist_df['seasons'].unique():
    season_df = hist_df[hist_df['seasons'] == season]
    season_best = best_df[best_df['seasons'] == season]
    
    base = alt.Chart(season_df).encode(
        x=alt.X("date_coeff:Q", title="date_coeff")
    )
    
    error_line = base.mark_boxplot().encode(
        y=alt.Y("Error:Q", scale=alt.Scale(zero=False), title="Error")
    )
    
    mpd_line = base.mark_boxplot().encode(
        y=alt.Y("Mean%Diff:Q", scale=alt.Scale(zero=False), title="Mean % Difference")
    )
    
    rules = alt.Chart(season_best).mark_rule(strokeDash=[6,4], color='red').encode(
        x=alt.X("date_coeff:Q", scale=alt.Scale(zero=False))
    )
    
    season_chart = alt.vconcat(
        (error_line + rules).properties(width=300, height=200, title=f"Season {season} - Error"),
        (mpd_line + rules).properties(width=300, height=200, title=f"Season {season} - Mean % Diff")
    )
    
    charts.append(season_chart)

chart = alt.hconcat(*charts)
chart

alt.HConcatChart(...)

In [18]:
import re

def Massey_weighted_optimized(league, home_win_coeff, away_win_coeff, margin_coeff, date_coeff):
    # Extract team names
    hTeam = league['home_team']
    aTeam = league['away_team']
    teams = sorted(set(hTeam).union(set(aTeam)))
    totalTeams = len(teams)

    # Map team names to indices
    teamIndex = {team: idx for idx, team in enumerate(teams)}

    # Weight vector (combined weighting mode 4)
    w = np.ones(len(league))

    # Weighting mode 4 logic (optimized)
    def margin(s):
        s = str(s).replace('–', '-')
        try:
            x, y = [int(t) for t in s.split('-')]
            diff = abs(x - y)
            return min(3, (1 + diff**margin_coeff)/2)
        except:
            return 1
    w_margin = league['score'].apply(margin).to_numpy()
    w_margin_norm = w_margin / np.mean(w_margin)

    # Home/Away weighting
    w_home_away = np.ones(len(league))
    for i in range(len(league)):
        row = league.iloc[i]
        home, away = row['home_team'], row['away_team']
        score_txt = str(row['score']).replace('–', '-')
        try:
            hScore, aScore = [int(t) for t in score_txt.split('-')]
            if hScore > aScore:
                w_home_away[i] = home_win_coeff  # home win: less credit
            elif aScore > hScore:
                w_home_away[i] = away_win_coeff  # away win: more credit
        except:
            continue

    latest_week = league['gameweek'].max()
    w_date_raw = (league['gameweek'] / latest_week).to_numpy() ** date_coeff
    w_date = w_date_raw / np.mean(w_date_raw)  # Normalize to mean=1

    # Combine weights (e.g., multiply them)
    w = w_margin_norm * w_home_away * w_date

    # Massey matrix and score vector
    M = np.zeros((totalTeams, totalTeams))
    b = np.zeros(totalTeams)

    # Process each game
    for i in range(len(league)):
        row = league.iloc[i]
        home = row['home_team']
        away = row['away_team']

        try:
            score = re.split(r'[-–]', row['score'])
            hScore = int(score[0])
            aScore = int(score[1])
        except:
            continue  # Skip if invalid score

        hIndex = teamIndex[home]
        aIndex = teamIndex[away]
        diff = hScore - aScore

        # Update matrix
        M[hIndex, hIndex] += w[i]
        M[aIndex, aIndex] += w[i]
        M[hIndex, aIndex] -= w[i]
        M[aIndex, hIndex] -= w[i]

        # Update score difference vector
        b[hIndex] += w[i] * diff
        b[aIndex] -= w[i] * diff

    # Massey system adjustment (last row to all 1's, last b to 0)
    M[-1, :] = 1
    b[-1] = 0

    # Solve for ratings
    r = np.linalg.solve(M, b)

    # Output results
    team_ratings = {team: r[idx] for team, idx in teamIndex.items()}
    sorted_teams = sorted(team_ratings, key=team_ratings.get, reverse=True)

    return team_ratings, sorted_teams

# Define Constant
best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_date_coeff = None
best_margin_coeff = None

In [19]:
history = []

try_num = 0

best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_margin_coeff = None

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
                # Run Massey with the current coefficients
                team_ratings, sorted_teams = Massey_weighted_optimized(train_liga_f_2022_2023, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2223)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2022/23',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
                })

                # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2223)
                    print("--------------------------------------------------------------------------------")


print("\nMassey Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=5.8310, Mean%Diff=0.0965
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Real Sociedad', 'Sevilla', 'Athletic Club', 'Valencia', 'Tenerife', 'Levante Planas', 'Sporting Huelva', 'Real Betis', 'Alavés', 'Alhama', 'Villarreal']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Tenerife', 'Sevilla', 'Real Sociedad', 'Valencia', 'Athletic Club', 'Levante Planas', 'Real Betis', 'Sporting Huelva', 'Villarreal', 'Alhama', 'Alavés']
--------------------------------------------------------------------------------

Massey Optimization Complete:
Best Home Win Coefficient: 0.25
Best Away Win Coefficient: 0.25
Best Margin Coefficient: 0.00
Best Date Coefficient: 0.00
Best Error: 5.8310
Best Mean Percentage Difference: 0.0965


In [20]:

try_num = 0

best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_margin_coeff = None

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
                # Run Massey with the current coefficients
                team_ratings, sorted_teams = Massey_weighted_optimized(train_liga_f_2023_2024, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2324)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2023/24',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
                })

                # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2324)
                    print("--------------------------------------------------------------------------------")


print("\nMassey Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=9.1652, Mean%Diff=0.1693
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Madrid CFF', 'Levante', 'Sevilla', 'Real Sociedad', 'Athletic Club', 'Tenerife', 'Levante Planas', 'Granada', 'Valencia', 'Villarreal', 'Sporting Huelva', 'Eibar', 'Real Betis']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Levante', 'Madrid CFF', 'Athletic Club', 'Sevilla', 'Real Sociedad', 'Tenerife', 'Eibar', 'Real Betis', 'Valencia', 'Levante Planas', 'Granada', 'Villarreal', 'Sporting Huelva']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.50, Error=8.1240, Mean%Diff=0.1308
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Levante', 'Madrid CFF', 'Sevilla', 'Real Sociedad', 'Athletic Club', 'Tenerife', 'Levante Planas', 'Granada

In [21]:

try_num = 0

best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_margin_coeff = None

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
                # Run Massey with the current coefficients
                team_ratings, sorted_teams = Massey_weighted_optimized(train_liga_f_2024_2025, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueLigaF2425)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2024/25',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
                })

                # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueLigaF2425)
                    print("--------------------------------------------------------------------------------")


print("\nMassey Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")

New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=6.8557, Mean%Diff=0.1328
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Real Sociedad', 'Athletic Club', 'Tenerife', 'Granada', 'Eibar', 'Levante Badalona', 'Sevilla', 'Espanyol', 'Madrid CFF', 'Real Betis', 'Dep La Coruña', 'Levante', 'Valencia']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Athletic Club', 'Granada', 'Tenerife', 'Real Sociedad', 'Eibar', 'Sevilla', 'Madrid CFF', 'Espanyol', 'Levante', 'Lavante Badalona', 'Dep La Coruña', 'Valencia', 'Real Betis']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.50, Date Coeff=0.00, Error=6.4807, Mean%Diff=0.1393
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Atlético Madrid', 'Real Sociedad', 'Athletic Club', 'Tenerife', 'Granada', 'Levante Badalona', 'Eibar', 'Sevilla', 'Madrid CFF', '

In [ ]:
# hist_df = pd.DataFrame(history)

# # best try per season (min Error, tie-break min Mean%Diff)
# best_df = (hist_df.sort_values(["seasons", "Error", "Mean%Diff"])
#                   .groupby("seasons", as_index=False)
#                   .first()[["seasons", "try"]])

# base = alt.Chart(hist_df).encode(
#     x=alt.X("try:Q", title="Try number"),
#     color=alt.Color("seasons:N", title="Season")
# )

# error_line = base.mark_line().encode(
#     y=alt.Y("Error:Q", scale=alt.Scale(zero=False),title="Error")
# )

# mpd_line = base.mark_line().encode(
#     y=alt.Y("Mean%Diff:Q", scale=alt.Scale(zero=False), title="Mean % Difference")
# )

# rules = alt.Chart(best_df).mark_rule(strokeDash=[6,4]).encode(
#     x="try:Q",
#     color=alt.Color("seasons:N", title="Season")
# )

# chart = alt.vconcat(
#     (error_line + rules).properties(width=600, height=200, title="Error over tries"),
#     (mpd_line + rules).properties(width=600, height=200, title="Mean % Difference over tries")
# ).resolve_scale(color="shared")

# chart

alt.VConcatChart(...)

In [23]:
from scipy import stats

vars_to_mode = ["home_coeff", "away_coeff", "margin_coeff", "date_coeff"]

def mode_at_min_mpd(df):
    min_mpd = df["Mean%Diff"].min()
    best_rows = df[df["Mean%Diff"] == min_mpd]
    
    mode_values = {}
    for col in vars_to_mode:
        m = best_rows[col].mode()
        # if multiple modes, take the first (deterministic)
        mode_values[col] = float(m.iloc[0])
    
    return pd.Series(mode_values)

# per-season
season_result = (
    hist_df
    .groupby("seasons")
    .apply(mode_at_min_mpd)
)

# ALL seasons
all_result = mode_at_min_mpd(hist_df).to_frame(name="ALL").T

# same orientation as before
final_table = pd.concat([season_result, all_result]).T

final_table


C:\Users\Naufal\AppData\Local\Temp\ipykernel_17852\2341830475.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


,2022/23,2023/24,2024/25,ALL
home_coeff,0.25,0.25,0.25,0.25
away_coeff,0.25,0.25,0.50,0.25
margin_coeff,0.00,0.00,0.00,0.00
date_coeff,0.00,0.50,0.50,0.00


In [24]:
from scipy import stats

vars_to_mode = ["home_coeff", "away_coeff", "margin_coeff", "date_coeff"]

def mode_at_min_mpd(df):
    min_mpd = df["Error"].min()
    best_rows = df[df["Error"] == min_mpd]
    
    mode_values = {}
    for col in vars_to_mode:
        m = best_rows[col].mode()
        # if multiple modes, take the first (deterministic)
        mode_values[col] = float(m.iloc[0])
    
    return pd.Series(mode_values)

# per-season
season_result = (
    hist_df
    .groupby("seasons")
    .apply(mode_at_min_mpd)
)

# ALL seasons
all_result = mode_at_min_mpd(hist_df).to_frame(name="ALL").T

# same orientation as before
final_table = pd.concat([season_result, all_result]).T

final_table


C:\Users\Naufal\AppData\Local\Temp\ipykernel_17852\2001641750.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


,2022/23,2023/24,2024/25,ALL
home_coeff,0.25,0.25,0.25,0.25
away_coeff,0.25,1.75,0.25,0.25
margin_coeff,0.00,0.50,0.50,0.00
date_coeff,0.00,1.00,0.00,0.00


In [25]:
hist_df.head(5)

,try,seasons,home_coeff,away_coeff,margin_coeff,date_coeff,Error,Mean%Diff
0,1,2022/23,0.25,0.25,0.0,0.0,5.830952,0.096549
1,2,2022/23,0.25,0.25,0.5,0.0,6.480741,0.127611
2,3,2022/23,0.25,0.25,1.0,0.0,6.928203,0.136986
3,4,2022/23,0.25,0.25,0.0,0.5,6.164414,0.134035
4,5,2022/23,0.25,0.25,0.5,0.5,7.745967,0.167273


In [26]:
# hist_df.to_csv("resultsMassey.csv", index=False)

In [27]:
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LinearRegression
# import pandas as pd
# import numpy as np

# def standardized_mlr_effect(df, y_col, covariates):
#     """
#     Returns standardized MLR coefficients as a pd.Series
#     """
#     X = df[covariates].values
#     y = df[y_col].values

#     # guard against degenerate seasons
#     if X.shape[0] < len(covariates):
#         return pd.Series(np.nan, index=covariates)

#     X_std = StandardScaler().fit_transform(X)

#     model = LinearRegression()
#     model.fit(X_std, y)

#     return pd.Series(model.coef_, index=covariates)
